# 03 – Water Quality Classification
**AquaML Portfolio Project**

Multi-class classification of water quality status (Good / Fair / Poor) from common physicochemical parameters.

**Models**: Random Forest, XGBoost, LightGBM  
**Focus**: class imbalance handling, feature importance, calibration awareness


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from src.evaluation import classification_metrics
from src.visualization import plot_confusion_matrix, plot_feature_importance


In [ ]:
wq = pd.read_csv(ROOT / "data/sample/water_quality.csv")
print(wq["quality"].value_counts(normalize=True).round(3))
wq.head()


In [ ]:
feature_cols = ["ph", "turbidity_ntu", "dissolved_oxygen_mgl", "bod_mgl", "nitrate_mgl", "temp_c"]
X = wq[feature_cols].values
le = LabelEncoder()
y = le.fit_transform(wq["quality"])
class_names = list(le.classes_)
print("Classes:", class_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)


## Train models

In [ ]:
models = {
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42, n_jobs=-1, eval_metric="mlogloss"),
    "LightGBM": LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, class_weight="balanced", random_state=42, verbose=-1),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    metrics = classification_metrics(y_test, pred)
    results[name] = metrics
    print(f"\n=== {name} ===")
    print(metrics)
    print(classification_report(y_test, pred, target_names=class_names, digits=3))


In [ ]:
# Confusion matrix for best model (LightGBM or RF)
best_name = max(results, key=lambda k: results[k]["f1"])
best_model = models[best_name]
pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, pred)
plot_confusion_matrix(cm, class_names, title=f"{best_name} – Confusion Matrix")
plt.show()


In [ ]:
# Feature importance
if hasattr(best_model, "feature_importances_"):
    plot_feature_importance(best_model.feature_importances_, feature_cols, title=f"{best_name} Feature Importance")
    plt.show()


## Takeaways
- Physicochemical parameters are highly informative for quality classification.
- Tree-based ensembles work very well out of the box.
- Always check per-class metrics – “Poor” water quality is often the minority but most important class.
- Feature importance helps domain experts trust and debug the model.
